### Check GPU availability and display GPU information

In [ ]:
gpu_info = !nvidia-smi
gpu_info = "\n".join(gpu_info)
if gpu_info.find("failed") >= 0:
    print("Not connected to a GPU")
else:
    print(gpu_info)


Fri Apr  3 16:06:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Install required Python packages (datasets, transformers, torchaudio, jiwer, accelerate, evaluate)

In [ ]:
%%capture
!pip install datasets==3.6.0
!pip install transformers==4.57.1
!pip install torchaudio
!pip install jiwer
!pip install accelerate -U
!pip install evaluate


### Install system dependencies and clone the KenLM language model toolkit

In [ ]:
!sudo apt-get update
!sudo apt-get install build-essential libboost-all-dev cmake zlib1g-dev libbz2-dev liblzma-dev
!sudo apt-get install libboost-all-dev libeigen3-dev
!git clone https://github.com/kpu/kenlm
%cd kenlm
!pip install e .

Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,482 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packag

### Build and install KenLM from source

In [ ]:
!mkdir build
%cd build
!cmake ..
!make -j 4
!sudo make install

mkdir: cannot create directory ‘build’: File exists
/content/kenlm/build
-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
CMake Warning (dev) at CMakeLists.txt:101 (find_package):
  Policy CMP0167 is not set: The FindBoost module is removed.  Run "cmake
  --help-policy CMP0167" for policy details.  Use the cmake_policy command to
  set the policy and suppress this warning.

This warning is for project developers.  Use -Wno-dev to suppress it.

-- Found Boost: /usr/lib/x86_64-linux-gnu/cmake/Boost-1.74.0/BoostConfig.cmake (found suitable v

### Add KenLM binaries to the system PATH and verify installation

In [ ]:
import os
os.environ['PATH'] += ":/content/kenlm/build/bin"  # Use your exact path
!echo $PATH  # Verify
!which lmplz  # Test


/opt/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin:/content/kenlm/build/bin
/usr/local/bin/lmplz


### Load the Dhivehi (Maldivian) Common Voice dataset (train+validation and test splits)

In [ ]:
from datasets import load_dataset

common_voice_train = load_dataset("fsicoli/common_voice_22_0", "dv", split="train+validation", trust_remote_code=True)
common_voice_test = load_dataset("fsicoli/common_voice_22_0", "dv", split="test", trust_remote_code=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

common_voice_22_0.py: 0.00B [00:00, ?B/s]

languages.py: 0.00B [00:00, ?B/s]

release_stats.py: 0.00B [00:00, ?B/s]

...


Reading metadata...: 2222it [00:00, 116951.44it/s]


Generating other split: 0 examples [00:00, ? examples/s]


Reading metadata...: 0it [00:00, ?it/s]
Reading metadata...: 15104it [00:00, 79405.43it/s]


Generating invalidated split: 0 examples [00:00, ? examples/s]


Reading metadata...: 1652it [00:00, 110437.99it/s]


### Remove unused metadata columns (accent, age, client_id, etc.) from train and test sets and display random samples

In [ ]:
common_voice_train = common_voice_train.remove_columns(
    ["accent", "age", "client_id", "down_votes", "gender", "locale", "segment", "up_votes"]
)
common_voice_test = common_voice_test.remove_columns(
    ["accent", "age", "client_id", "down_votes", "gender", "locale", "segment", "up_votes"]
)

from datasets import ClassLabel
import random
import pandas as pd
from IPython.display import display, HTML

def show_random_elements(dataset, num_examples=10):
    assert num_examples <= len(dataset), "Can't pick more elements than there are in the dataset."
    picks = []
    for _ in range(num_examples):
        pick = random.randint(0, len(dataset) - 1)
        while pick in picks:
            pick = random.randint(0, len(dataset) - 1)
        picks.append(pick)
    df = pd.DataFrame(dataset[picks])
    display(HTML(df.to_html()))

# Select the 'train' split before calling show_random_elements
show_random_elements(common_voice_train.remove_columns(["path", "audio"]), num_examples=10)


,sentence,variant
0,ރަށުގެ ހުރިހާ ރައްޔިތުންހެން އެއްވެ ތިވި,
1,ކައްކައި ކެއުމަށް ބޭނުންކުރާ ބައެއް ތަކެތި ގަތުމަށް,
2,އެހެންނަމަވެސް ވަރިހަމަ,
3,މިއީ ވަރަށް ސީރިއަސް ކަމެއް ކަން ދަންނަނީ އަދި މާ މަދު ބަޔެއް,
4,އެތާނގައި ތިބި ކުދިންތަކާއި ބެލެނިވެރިންތައް އެހާ ގިނަވެފައި މީޑިއާތަކާއި ސަރުކާރުގެ ބޮޑުންވެސް ތިބުމުން ކަންނޭނގެ,
5,ފިރިހެނުންތައް އެތިބީ އެދޯނީގެ ހިޔާގަނޑު މައްޗަށާއި ގެގަނޑުމައްޗަށް އަރާފަ,
6,އޭނާ ދެންނެވީ ތިޔަ ވިދާޅުވީ ތެދެއް,
7,ބައްޕަ ހިތްހަމަނުޖެހިފައިހުރެ ނުކުމެގެންދިޔަ މަންޒަރު ބަލަން އަހަރެން އިނީ ފެންކަޅިވެފަ,
8,ދެން ދެންނެވީ އިޙުސާނާ ބެހޭ ގޮތުން ތިމަންނާއަށް ޚަބަރު ދެއްވާ,
9,އެންމެ ރަށަކުން ވެސް ނޫން,


### Define and apply text cleaning: remove punctuation, special characters, and filter out Arabic script rows

In [ ]:
import re

chars_to_remove_regex = r"""[\,\?\.\!\-\;\:\"\“\%\‘\”\�\'\»\«\–\’]"""

def remove_special_characters(batch):
    batch["sentence"] = re.sub(chars_to_remove_regex, "", batch["sentence"]).lower().strip()
    return batch

common_voice_train = common_voice_train.map(remove_special_characters)
common_voice_test  = common_voice_test.map(remove_special_characters)

arabic_pattern = re.compile(r"[\u0600-\u06FF\u0750-\u077F\u08A0-\u08FF\uFB50-\uFDFF\uFE70-\uFEFF]")

def remove_arabic_rows(batch):
    return not bool(arabic_pattern.search(batch["sentence"]))

common_voice_train = common_voice_train.filter(remove_arabic_rows)
common_voice_test  = common_voice_test.filter(remove_arabic_rows)

Map:   0%|          | 0/4897 [00:00<?, ? examples/s]

Map:   0%|          | 0/2222 [00:00<?, ? examples/s]

Filter:   0%|          | 0/4897 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2222 [00:00<?, ? examples/s]

### Extract the unique character vocabulary from train and test sets

In [ ]:
def extract_all_chars(batch):
    all_text = " ".join(batch["sentence"])
    vocab = list(set(all_text))
    return {"vocab": [vocab], "all_text": [all_text]}

vocab_train = common_voice_train.map(
    extract_all_chars,
    batched=True,
    batch_size=-1,
    keep_in_memory=True,
    remove_columns=common_voice_train.column_names,
)
vocab_test = common_voice_test.map(
    extract_all_chars,
    batched=True,
    batch_size=-1,
    keep_in_memory=True,
    remove_columns=common_voice_test.column_names,
)

vocab_list = list(set(vocab_train["vocab"][0]) | set(vocab_test["vocab"][0]))
vocab_dict = {v: k for k, v in enumerate(sorted(vocab_list))}
print(vocab_dict)

Map:   0%|          | 0/4600 [00:00<?, ? examples/s]

Map:   0%|          | 0/2077 [00:00<?, ? examples/s]

{' ': 0, 'ހ': 1, 'ށ': 2, 'ނ': 3, 'ރ': 4, 'ބ': 5, 'ޅ': 6, 'ކ': 7, 'އ': 8, 'ވ': 9, 'މ': 10, 'ފ': 11, 'ދ': 12, 'ތ': 13, 'ލ': 14, 'ގ': 15, 'ޏ': 16, 'ސ': 17, 'ޑ': 18, 'ޒ': 19, 'ޓ': 20, 'ޔ': 21, 'ޕ': 22, 'ޖ': 23, 'ޗ': 24, 'ޘ': 25, 'ޙ': 26, 'ޚ': 27, 'ޛ': 28, 'ޝ': 29, 'ޞ': 30, 'ޟ': 31, 'ޠ': 32, 'ޡ': 33, 'ޢ': 34, 'ޣ': 35, 'ޤ': 36, 'ޥ': 37, 'ަ': 38, 'ާ': 39, 'ި': 40, 'ީ': 41, 'ު': 42, 'ޫ': 43, 'ެ': 44, 'ޭ': 45, 'ޮ': 46, 'ޯ': 47, 'ް': 48}


### Remove any remaining Latin characters from transcriptions and re-extract the character vocabulary

In [ ]:
def remove_latin_characters(batch):
    batch["sentence"] = re.sub(r"[a-z]+", "", batch["sentence"])
    return batch

common_voice_train = common_voice_train.map(remove_latin_characters)
common_voice_test = common_voice_test.map(remove_latin_characters)

vocab_train = common_voice_train.map(
    extract_all_chars,
    batched=True,
    batch_size=-1,
    keep_in_memory=True,
    remove_columns=common_voice_train.column_names,
)
vocab_test = common_voice_test.map(
    extract_all_chars,
    batched=True,
    batch_size=-1,
    keep_in_memory=True,
    remove_columns=common_voice_test.column_names,
)

vocab_list = list(set(vocab_train["vocab"][0]) | set(vocab_test["vocab"][0]))
vocab_dict = {v: k for k, v in enumerate(sorted(vocab_list))}
print(vocab_dict)


Map:   0%|          | 0/4600 [00:00<?, ? examples/s]

Map:   0%|          | 0/2077 [00:00<?, ? examples/s]

Map:   0%|          | 0/4600 [00:00<?, ? examples/s]

Map:   0%|          | 0/2077 [00:00<?, ? examples/s]

{' ': 0, 'ހ': 1, 'ށ': 2, 'ނ': 3, 'ރ': 4, 'ބ': 5, 'ޅ': 6, 'ކ': 7, 'އ': 8, 'ވ': 9, 'މ': 10, 'ފ': 11, 'ދ': 12, 'ތ': 13, 'ލ': 14, 'ގ': 15, 'ޏ': 16, 'ސ': 17, 'ޑ': 18, 'ޒ': 19, 'ޓ': 20, 'ޔ': 21, 'ޕ': 22, 'ޖ': 23, 'ޗ': 24, 'ޘ': 25, 'ޙ': 26, 'ޚ': 27, 'ޛ': 28, 'ޝ': 29, 'ޞ': 30, 'ޟ': 31, 'ޠ': 32, 'ޡ': 33, 'ޢ': 34, 'ޣ': 35, 'ޤ': 36, 'ޥ': 37, 'ަ': 38, 'ާ': 39, 'ި': 40, 'ީ': 41, 'ު': 42, 'ޫ': 43, 'ެ': 44, 'ޭ': 45, 'ޮ': 46, 'ޯ': 47, 'ް': 48}


### Finalize the vocabulary: replace space with pipe delimiter, add [UNK] and [PAD] tokens, and save vocab.json

In [ ]:
# use "|" as word delimiter instead of space
vocab_dict["|"] = vocab_dict[" "]
del vocab_dict[" "]

# add special tokens
vocab_dict["[UNK]"] = len(vocab_dict)
vocab_dict["[PAD]"] = len(vocab_dict)

print("Vocab size:", len(vocab_dict))

import json
with open("vocab.json", "w", encoding="utf-8") as vocab_file:
    json.dump(vocab_dict, vocab_file, ensure_ascii=False)


Vocab size: 51


### Initialize the CTC tokenizer, Wav2Vec2 feature extractor, and Wav2Vec2 processor

In [ ]:
from transformers import Wav2Vec2CTCTokenizer, Wav2Vec2FeatureExtractor, Wav2Vec2Processor

tokenizer = Wav2Vec2CTCTokenizer.from_pretrained(
    "./",
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|",
)

feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1,
    sampling_rate=16000,
    padding_value=0.0,
    do_normalize=True,
    return_attention_mask=True,
)

processor = Wav2Vec2Processor(
    feature_extractor=feature_extractor,
    tokenizer=tokenizer,
)


### Resample all audio to 16kHz and preview a random training sample

In [ ]:
from datasets import Audio

common_voice_train = common_voice_train.cast_column("audio", Audio(sampling_rate=16_000))
common_voice_test = common_voice_test.cast_column("audio", Audio(sampling_rate=16_000))

print(common_voice_train[0]["audio"])

import IPython.display as ipd
import numpy as np
import random

rand_int = random.randint(0, len(common_voice_train) - 1)
print("Target text:", common_voice_train[rand_int]["sentence"])
print("Input array shape:", common_voice_train[rand_int]["audio"]["array"].shape)
print("Sampling rate:", common_voice_train[rand_int]["audio"]["sampling_rate"])

ipd.Audio(
    data=common_voice_train[rand_int]["audio"]["array"],
    autoplay=True,
    rate=16000,
)


{'path': '/root/.cache/huggingface/datasets/downloads/extracted/e24cb6e0896730e3c68f4266eadc6dbf6150fbe071ba9968bf13481404180b10/dv_train_0/common_voice_dv_18580646.mp3', 'array': array([1.45519152e-11, 1.01863407e-10, 1.30967237e-10, ...,
       7.25846985e-06, 1.29183172e-06, 5.80571304e-06]), 'sampling_rate': 16000}
Target text: އެއާއެކު ރޫޝިންގެ މޫނުމަތިން ފެނިގެން ދިޔައީ ހައިރާންކަމުގެ ކުލަވަރު
Input array shape: (95616,)
Sampling rate: 16000


### Write all transcriptions to a text file for KenLM language model training

In [ ]:
with open("lm_corpus.txt", "w", encoding="utf-8") as f:
    for ex in common_voice_train:
        s = ex["sentence"].strip()
        if s:
            f.write(s + "\n")
    for ex in common_voice_test:
        s = ex["sentence"].strip()
        if s:
            f.write(s + "\n")

print("Wrote LM corpus to lm_corpus.txt")

Wrote LM corpus to lm_corpus.txt


### Define the dataset preparation function: extract audio input values and encode text labels, then apply to train and test sets

In [ ]:
def prepare_dataset(batch):
    audio = batch["audio"]

    batch["input_values"] = processor(
        audio["array"],
        sampling_rate=audio["sampling_rate"]
    ).input_values[0]

    batch["input_length"] = len(batch["input_values"])

    with processor.as_target_processor():
        batch["labels"] = processor(batch["sentence"]).input_ids

    return batch

common_voice_train = common_voice_train.map(
    prepare_dataset,
    remove_columns=common_voice_train.column_names,
)
common_voice_test = common_voice_test.map(
    prepare_dataset,
    remove_columns=common_voice_test.column_names,
)


Map:   0%|          | 0/4600 [00:00<?, ? examples/s]

Map:   0%|          | 0/2077 [00:00<?, ? examples/s]

### Define a custom data collator that pads input features and labels for CTC training

In [ ]:
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Union

@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_values": f["input_values"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]

        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt",
        )

        with self.processor.as_target_processor():
            labels_batch = self.processor.pad(
                label_features,
                padding=self.padding,
                return_tensors="pt",
            )

        labels = labels_batch["input_ids"].masked_fill(labels_batch["attention_mask"].ne(1), -100)
        batch["labels"] = labels

        return batch

### Train a trigram KenLM language model from the corpus

In [ ]:
!lmplz -o 3 < lm_corpus.txt > lm.arpa

=== 1/5 Counting and sorting n-grams ===
Reading /content/kenlm/build/lm_corpus.txt
----5---10---15---20---25---30---35---40---45---50---55---60---65---70---75---80---85---90---95--100
****************************************************************************************************
Unigram tokens 44171 types 17424
=== 2/5 Calculating and sorting adjusted counts ===
Chain sizes: 1:209088 2:15234823168 3:28565295104
Statistics:
1 17424 D1=0.746612 D2=1.12844 D3+=1.44662
2 40523 D1=0.916656 D2=1.1996 D3+=1.3213
3 42478 D1=0.969956 D2=1.42164 D3+=1.57538
Memory estimate for binary LM:
type      kB
probing 2138 assuming -p 1.5
probing 2444 assuming -r models -p 1.5
trie    1111 without quantization
trie     763 assuming -q 8 -b 8 quantization 
trie    1067 assuming -a 22 array pointer compression
trie     718 assuming -a 22 -q 8 -b 8 array pointer compression and quantization
=== 3/5 Calculating and sorting initial probabilities ===
Chain sizes: 1:209088 2:648368 3:849560
----5---10---15

### Install pyctcdecode library for CTC beam search decoding with language model support

In [ ]:
!pip install pyctcdecode

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.5/529.5 kB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 133.2 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2

### Build a CTC beam search decoder with KenLM language model integration

In [ ]:
from pyctcdecode import build_ctcdecoder

# get labels in ID order
vocab = processor.tokenizer.get_vocab()  # token -> id
id2token = {v: k for k, v in vocab.items()}
labels = [id2token[i] for i in range(len(id2token))]

# path to your trained KenLM model (arpa or binary)
kenlm_model_path = "lm.arpa"  # <-- change if you use another path/name

decoder = build_ctcdecoder(
    labels=labels,
    kenlm_model_path=kenlm_model_path,
    alpha=0.5,  # tune on dev
    beta=1.0  # tune on dev
)


### Define the evaluation metrics function using LM-aided beam search decoding to compute WER and CER

In [ ]:
import evaluate
import torch

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    pred_logits = pred.predictions
    label_ids = pred.label_ids

    # Convert logits to probabilities
    probs = torch.softmax(torch.tensor(pred_logits), dim=-1).numpy()

    # Decode with LM
    pred_str = [decoder.decode(p, beam_width=64) for p in probs]

    # Replace -100 with pad_token_id
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    label_str = processor.batch_decode(label_ids, group_tokens=False)

    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    cer = cer_metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer, "cer": cer}

### Load the continually pre-trained Wav2Vec2 model with a new CTC head for Dhivehi fine-tuning

In [ ]:
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
import torch
import numpy as np

checkpoint_dir = "/content/drive/MyDrive/final_model65"

model = Wav2Vec2ForCTC.from_pretrained(checkpoint_dir).to("cuda")
processor = Wav2Vec2Processor.from_pretrained(checkpoint_dir)

model.eval()

all_pred_raw = []        # ASR + LM only
all_pred_corrected = []  # ASR + LM + T5 spell corrector
all_refs = []            # ground-truth text

for i, ex in enumerate(common_voice_test):
    # 1. Forward pass on stored input_features from your prepared dataset
    input_feats = torch.tensor(ex["input_features"]).unsqueeze(0).to(model.device)
    with torch.no_grad():
        logits = model(input_feats).logits

    # 2. Convert logits to probabilities
    probs = torch.softmax(logits[0], dim=-1).cpu().numpy()

    # 3. Decode with LM (wav2vec2 + CTC + KenLM)
    pred_text = decoder.decode(probs)
    all_pred_raw.append(pred_text)

    # 5. Reference text from labels
    ref_text = processor.decode(ex["labels"], group_tokens=False).lower()
    all_refs.append(ref_text)

# 6. Compute WERs
wer_raw = wer_metric.compute(
    predictions=all_pred_raw,
    references=all_refs,
)
cer_raw = cer_metric.compute(predictions=all_pred_raw, references=all_refs)

print(f"WER (ASR + LM only)           : {wer_raw:.4f}")
print(f"CER (ASR + LM only)           : {cer_raw:.4f}")

# Optional: inspect a few examples
for idx in range(3):
    print("\n--- Example", idx, "---")
    print("REF :", all_refs[idx])
    print("RAW :", all_pred_raw[idx])

WER (ASR + LM only)           : 0.1289
CER (ASR + LM only)           : 0.027

--- Example 0 ---
REF : ޕިކަޕް މަރާމާތުކޮށްދޭނެ ފަރާތެއް ހޯދުން
RAW : ޕިކަޕް މަރާމާތުކޮށްދޭނެ ފަރާތެއް ހޯދުން

--- Example 1 ---
REF : އަންޖޫދާ މަޑުމަޑުން އެހެން ބުނެލުމުން ރިޝްކާ ވެސް އިތުރު ސުވާލެއް ނުކުރޭ
RAW : އަންޖޫދާ މަޑުމަޑުން އެހެން ބުނެލުމުން ރިޝްކާ ވެސް އިތުރު ސުވާލެއް ނުކުރޭ

--- Example 2 ---
REF : އެސްފިޔަތަކުން ވަނީ ނިދި ގެއްލިފަ
RAW : އެސްފިޔަތަކުން ވަނީ ނިދި ގެއްލިފަ


### Mount Google Drive for persistent storage

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
